# Public-Data Factor IC Demo

This notebook shows how to run a small factor Information Coefficient (IC) analysis with public market data. It is meant as a reproducibility and API demo, not as investment advice or a trading recommendation.

The notebook tries `yfinance` first. If public data download fails because of network, API, or environment limits, it falls back to the repository's synthetic panel so the workflow still runs end to end.

## Setup

If you are running this notebook from a fresh Colab runtime, uncomment the install cell below.

In [ ]:
# Uncomment in a fresh Colab runtime:
# !git clone https://github.com/initial-d/ml-quant-trading.git
# %pip install -q -e ml-quant-trading[dev]
# import sys
# sys.path.insert(0, "ml-quant-trading/src")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from mlquant.data import make_panel
from mlquant.features import compute_legacy_set

plt.style.use("seaborn-v0_8-whitegrid")

## Load A Small Public Universe

The default universe uses large, liquid US equities so most readers can download the data without proprietary credentials. You can replace these tickers with A-share symbols such as `000001.SZ` and `600000.SS` if your data provider supports them reliably.

In [ ]:
tickers = [
    "AAPL", "MSFT", "NVDA", "GOOGL", "AMZN",
    "META", "JPM", "XOM", "UNH", "LLY",
]

try:
    panel = make_panel(
        source="yfinance",
        tickers=tickers,
        start="2021-01-01",
        end="2025-01-01",
        device="cpu",
    )
    data_source = "yfinance"
except Exception as exc:
    print(f"Falling back to synthetic data because yfinance failed: {exc}")
    panel = make_panel(source="synthetic", n_stocks=80, n_dates=750, seed=7, device="cpu")
    data_source = "synthetic"

panel.assert_consistent()
print({
    "source": data_source,
    "dates": int(panel.n_dates),
    "stocks": int(panel.n_stocks),
    "start": str(pd.Timestamp(panel.dates[0]).date()),
    "end": str(pd.Timestamp(panel.dates[-1]).date()),
})

## Compute A Small Factor Set

We use a compact subset of the legacy factor registry to keep the notebook quick. The same pattern works with any factor names from `LEGACY_REGISTRY`.

In [ ]:
factor_names = (
    "best_001",
    "best_002",
    "original_001",
    "stock_001",
    "add_015",
    "old_042",
)

factors, factor_mask, names = compute_legacy_set(panel, names=factor_names)
print(factors.shape)
print(names)

## Calculate One-Day Forward Rank IC

IC is the cross-sectional correlation between today's factor values and a future return. Here we use Spearman rank correlation against one-day forward returns. This is a simple signal-quality diagnostic, not a full strategy backtest.

In [ ]:
def one_day_forward_returns(panel):
    fwd = torch.roll(panel.returns, shifts=-1, dims=0)
    fwd[-1] = 0.0
    tradable_next = panel.mask & torch.roll(panel.mask, shifts=-1, dims=0)
    tradable_next[-1] = False
    return fwd, tradable_next


def rank_ic_by_date(factor_values, returns, valid_mask, dates, names):
    rows = []
    values_np = factor_values.detach().cpu().numpy()
    returns_np = returns.detach().cpu().numpy()
    mask_np = valid_mask.detach().cpu().numpy()

    for t, date in enumerate(dates):
        row = {"date": pd.Timestamp(date)}
        for j, name in enumerate(names):
            m = mask_np[t]
            x = values_np[t, :, j]
            y = returns_np[t]
            ok = m & np.isfinite(x) & np.isfinite(y)
            row[name] = np.nan if ok.sum() < 3 else pd.Series(x[ok]).corr(pd.Series(y[ok]), method="spearman")
        rows.append(row)

    return pd.DataFrame(rows).set_index("date")


fwd_returns, fwd_mask = one_day_forward_returns(panel)
valid_mask = factor_mask & fwd_mask
ic = rank_ic_by_date(factors, fwd_returns, valid_mask, panel.dates, names)
ic.tail()

## Summarize IC

In [ ]:
summary = pd.DataFrame({
    "mean_ic": ic.mean(skipna=True),
    "median_ic": ic.median(skipna=True),
    "ic_std": ic.std(skipna=True),
    "positive_rate": (ic > 0).mean(skipna=True),
    "observations": ic.count(),
}).sort_values("mean_ic", ascending=False)

summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

summary["mean_ic"].plot(kind="bar", ax=axes[0], color="#4477aa")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Mean one-day rank IC")
axes[0].set_ylabel("Spearman IC")

ic.rolling(60, min_periods=20).mean().plot(ax=axes[1], linewidth=1.5)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("60-day rolling rank IC")
axes[1].set_ylabel("Spearman IC")

plt.tight_layout()

## Inspect One Cross-Section

The scatter below shows factor rank versus next-day return for the date with the largest number of valid observations. With a tiny public universe this chart is noisy, but it is useful for checking whether the pipeline is wired correctly.

In [ ]:
valid_counts = valid_mask.detach().cpu().numpy().sum(axis=1)
t = int(valid_counts.argmax())
chosen_factor = summary.index[0]
j = names.index(chosen_factor)

x = pd.Series(factors[t, :, j].detach().cpu().numpy(), index=panel.stocks, name=chosen_factor)
y = pd.Series(fwd_returns[t].detach().cpu().numpy(), index=panel.stocks, name="next_day_return")
m = pd.Series(valid_mask[t].detach().cpu().numpy(), index=panel.stocks).astype(bool)
cross_section = pd.concat([x, y], axis=1).loc[m]
cross_section["factor_rank"] = cross_section[chosen_factor].rank(pct=True)

ax = cross_section.plot.scatter(x="factor_rank", y="next_day_return", figsize=(7, 4), color="#4477aa")
ax.axhline(0, color="black", linewidth=1)
ax.set_title(f"{chosen_factor} rank vs next-day return on {pd.Timestamp(panel.dates[t]).date()}")
plt.tight_layout()

cross_section.sort_values("factor_rank", ascending=False).head(10)

## How To Extend This Notebook

- Replace `tickers` with your own public universe.
- Increase the date range for more IC observations.
- Add more factor names from `mlquant.features.LEGACY_REGISTRY`.
- Compare raw factors with neutralized factors by calling `compute_legacy_set(..., neutralize=False)`.
- Move from IC diagnostics to a full portfolio backtest only after documenting transaction costs, slippage, universe construction, and leakage controls.

For more context, see `docs/backtest_assumptions.md`.